In [ ]:
pip install biopython pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 39.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from Bio import Entrez
import time


Entrez.email = "naga7dna@gmail.com"
def search_pubmed(query, max_results=50):
    """Search PubMed and retrieve a list of PMIDs matching the query."""
    print(f"Searching PubMed for query: '{query}'...")
    handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results)
    record = Entrez.read(handle)
    handle.close()
    return record["IdList"]

def fetch_abstracts(id_list):
    """Fetch article metadata and abstracts using a list of PMIDs."""
    if not id_list:
        print("No papers found.")
        return pd.DataFrame()

    print(f"Fetching metadata for {len(id_list)} biomedical articles...")
    ids = ",".join(id_list)

    handle = Entrez.efetch(db="pubmed", id=ids, retmode="xml")
    records = Entrez.read(handle)
    handle.close()

    articles = []

    for pubmed_article in records['PubmedArticle']:
        try:
            citation = pubmed_article['MedlineCitation']
            article = citation['Article']
            pmid = citation['PMID']
            title = article.get('ArticleTitle', 'No title available')
            journal = article['Journal'].get('Title', 'No journal available')


            pub_date = article['Journal']['JournalIssue']['PubDate']
            year = pub_date.get('Year', pub_date.get('MedlineDate', 'Unknown'))

            abstract = 'No abstract available'
            if 'Abstract' in article:
                abstract_text = article['Abstract'].get('AbstractText', [])
                if abstract_text:
                    abstract = " ".join([str(text) for text in abstract_text])

            articles.append({
                'PMID': str(pmid),
                'Title': title,
                'Year': year,
                'Journal': journal,
                'Abstract': abstract,
                'URL': f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
            })
        except Exception as e:
            continue

    return pd.DataFrame(articles)

if __name__ == "__main__":

    SEARCH_QUERY = '("endometrial receptivity" OR "window of implantation") AND ("transcriptomics" OR "multi-omics" OR "RNA-Seq")'
    TOTAL_PAPERS = 1000

    pmid_list = search_pubmed(SEARCH_QUERY, max_results=TOTAL_PAPERS)
    df_results = fetch_abstracts(pmid_list)

    if not df_results.empty:
        output_csv = "cancer_reserch_papers.csv"
        df_results.to_csv(output_csv, index=False, encoding="utf-8")
        print(f"\nSuccessfully saved {len(df_results)} papers to '{output_csv}'!")
        print("\nFirst Paper Preview:")
        print(f"Title: {df_results.iloc[0]['Title']}")
        print(f"PMID: {df_results.iloc[0]['PMID']}")
        print(f"Abstract: {df_results.iloc[0]['Abstract'][:250]}...")

Searching PubMed for query: '("endometrial receptivity" OR "window of implantation") AND ("transcriptomics" OR "multi-omics" OR "RNA-Seq")'...
Fetching metadata for 135 biomedical articles...

Successfully saved 135 papers to 'cancer_reserch_papers.csv'!

First Paper Preview:
Title: NOS1 mediates LPS-induced oxidative stress and impaired endometrial receptivity in sheep during early pregnancy.
PMID: 42731421
Abstract: In sheep production, bacterial infections often cause embryo implantation failure and early pregnancy loss, resulting in substantial economic losses. These detrimental effects are closely associated with bacterial lipopolysaccharide (LPS)-induced dis...


==================================================

In [ ]:
import pandas as pd
from Bio import Entrez

Entrez.email = "naga.email@example.com"

def search_optimal_pubmed(query, max_results=100):
    """
    Search PubMed sorted by Best Match (relevance)
    to retrieve the most optimal papers.
    """
    print(f"Searching PubMed for top {max_results} optimal papers...")

    # Added sort="relevance"
    handle = Entrez.esearch(
        db="pubmed",
        term=query,
        retmax=max_results,
        sort="relevance"
    )
    record = Entrez.read(handle)
    handle.close()
    return record["IdList"]

def fetch_abstracts(id_list):
    if not id_list:
        print("No papers found.")
        return pd.DataFrame()

    print(f"Fetching metadata for {len(id_list)} highly relevant articles...")
    ids = ",".join(id_list)

    handle = Entrez.efetch(db="pubmed", id=ids, retmode="xml")
    records = Entrez.read(handle)
    handle.close()

    articles = []
    for pubmed_article in records['PubmedArticle']:
        try:
            citation = pubmed_article['MedlineCitation']
            article = citation['Article']
            pmid = citation['PMID']
            title = article.get('ArticleTitle', 'No title available')
            journal = article['Journal'].get('Title', 'No journal available')

            pub_date = article['Journal']['JournalIssue']['PubDate']
            year = pub_date.get('Year', pub_date.get('MedlineDate', 'Unknown'))

            abstract = 'No abstract available'
            if 'Abstract' in article:
                abstract_text = article['Abstract'].get('AbstractText', [])
                if abstract_text:
                    abstract = " ".join([str(text) for text in abstract_text])

            articles.append({
                'PMID': str(pmid),
                'Title': title,
                'Year': year,
                'Journal': journal,
                'Abstract': abstract,
                'URL': f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
            })
        except Exception as e:
            continue

    return pd.DataFrame(articles)

if __name__ == "__main__":

    # PubMed query using
    SEARCH_QUERY = (
        '("primary and metastatic"[tiab] OR "paired primary"[tiab] OR "matched primary"[tiab] OR "paired samples"[tiab] OR "metastatic progression"[tiab]) '
        'AND ("cancer"[tiab] OR "carcinoma"[tiab] OR "tumor"[tiab]) '
        'AND ("RNA-Seq"[tiab] OR "transcriptomics"[tiab] OR "multi-omics"[tiab] OR "scRNA-seq"[tiab] OR "single-cell"[tiab]) '
        'NOT "Review"[Publication Type]'
    )


    TOP_N_PAPERS = 100

    pmid_list = search_optimal_pubmed(SEARCH_QUERY, max_results=TOP_N_PAPERS)
    df_results = fetch_abstracts(pmid_list)

    if not df_results.empty:
        output_csv = "optimal_paired_metastatic_papers.csv"
        df_results.to_csv(output_csv, index=False, encoding="utf-8")
        print(f"\nSuccessfully saved top {len(df_results)} optimal papers to '{output_csv}'!")
        print("\nTop Ranked Paper Preview:")
        print(f"Title: {df_results.iloc[0]['Title']}")
        print(f"PMID: {df_results.iloc[0]['PMID']}")

Searching PubMed for top 100 optimal papers...
Fetching metadata for 100 highly relevant articles...

Successfully saved top 100 optimal papers to 'optimal_paired_metastatic_papers.csv'!

Top Ranked Paper Preview:
Title: Single-Cell Transcriptomic Analysis of Primary and Metastatic Tumor Ecosystems in Head and Neck Cancer.
PMID: 29198524


BRCA breast cancer papers

In [ ]:
import pandas as pd
from Bio import Entrez

Entrez.email = "nagatest@gmail.com"

def search_optimal_pubmed(query, max_results=100):
    """
    Search PubMed sorted by Best Match (relevance)
    to pull optimal BRCA breast cancer paired primary-metastatic studies.
    """
    print(f"Searching PubMed for top {max_results} BRCA breast cancer papers...")

    # Best Match ranking (relevance)
    handle = Entrez.esearch(
        db="pubmed",
        term=query,
        retmax=max_results,
        sort="relevance"
    )
    record = Entrez.read(handle)
    handle.close()
    return record["IdList"]

def fetch_abstracts(id_list):
    if not id_list:
        print("No papers found matching criteria.")
        return pd.DataFrame()

    print(f"Fetching metadata for {len(id_list)} articles...")
    ids = ",".join(id_list)

    handle = Entrez.efetch(db="pubmed", id=ids, retmode="xml")
    records = Entrez.read(handle)
    handle.close()

    articles = []
    for pubmed_article in records['PubmedArticle']:
        try:
            citation = pubmed_article['MedlineCitation']
            article = citation['Article']
            pmid = citation['PMID']
            title = article.get('ArticleTitle', 'No title available')
            journal = article['Journal'].get('Title', 'No journal available')

            pub_date = article['Journal']['JournalIssue']['PubDate']
            year = pub_date.get('Year', pub_date.get('MedlineDate', 'Unknown'))

            abstract = 'No abstract available'
            if 'Abstract' in article:
                abstract_text = article['Abstract'].get('AbstractText', [])
                if abstract_text:
                    abstract = " ".join([str(text) for text in abstract_text])

            articles.append({
                'PMID': str(pmid),
                'Title': title,
                'Year': year,
                'Journal': journal,
                'Abstract': abstract,
                'URL': f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
            })
        except Exception as e:
            continue

    return pd.DataFrame(articles)

if __name__ == "__main__":

    # Query: Paired Primary/Metastatic + Breast Cancer + BRCA + Omics
    SEARCH_QUERY = (
        '("primary and metastatic"[tiab] OR "paired primary"[tiab] OR "matched primary"[tiab] OR "paired samples"[tiab] OR "primary-metastatic"[tiab] OR "paired primary and recurrent"[tiab]) '
        'AND ("breast cancer"[tiab] OR "breast carcinoma"[tiab] OR "breast tumor"[tiab]) '
        'AND ("BRCA"[tiab] OR "BRCA1"[tiab] OR "BRCA2"[tiab] OR "BRCA-mutated"[tiab] OR "BRCAness"[tiab]) '
        'AND ("RNA-Seq"[tiab] OR "transcriptomics"[tiab] OR "multi-omics"[tiab] OR "scRNA-seq"[tiab] OR "single-cell"[tiab]) '
        'NOT "Review"[Publication Type]'
    )

    TOP_N_PAPERS = 100

    pmid_list = search_optimal_pubmed(SEARCH_QUERY, max_results=TOP_N_PAPERS)
    df_results = fetch_abstracts(pmid_list)

    if not df_results.empty:
        output_csv = "brca_breast_cancer_paired_metastatic.csv"
        df_results.to_csv(output_csv, index=False, encoding="utf-8")
        print(f"\nSuccessfully saved top {len(df_results)} BRCA breast cancer papers to '{output_csv}'!")
        print("\nTop Ranked Paper Preview:")
        print(f"Title: {df_results.iloc[0]['Title']}")
        print(f"PMID: {df_results.iloc[0]['PMID']}")
        print(f"Abstract: {df_results.iloc[0]['Abstract'][:250]}...")

Searching PubMed for top 100 BRCA breast cancer papers...
Fetching metadata for 4 articles...

Successfully saved top 4 BRCA breast cancer papers to 'brca_breast_cancer_paired_metastatic.csv'!

Top Ranked Paper Preview:
Title: A Targetable EGFR-Dependent Tumor-Initiating Program in Breast Cancer.
PMID: 29091754
Abstract: Therapies targeting epidermal growth factor receptor (EGFR) have variable and unpredictable responses in breast cancer. Screening triple-negative breast cancer (TNBC) patient-derived xenografts (PDXs), we identify a subset responsive to EGFR inhibiti...
